# IMDb Top Netflix Movies and TV Shows

Dataset link: https://www.kaggle.com/datasets/bharatnatrayn/movies-dataset-for-feature-extracion-prediction?select=movies.csv

## Contents
- [Imports and Loading](#imports-and-loading)
- [Overview](#overview)
  - [Whitespace and Column Cleaning](#whitespace-and-column-cleaning)
  - [Duplicate Handling](#duplicate-handling)
  - [Overall Insights](#overall-insights)
- [Data Formatting and Type Conversion](#data-formatting-and-type-conversion)
  - [Formatting Strings](#formatting-strings)
  - [Numerical Data Conversion](#numerical-data-conversion)
  - [Conclusion](#conclusion)
- [ROMI Approach](#romi-approach)
  - [Relation](#relation)
  - [Outlier](#outlier)
  - [Mismatch](#mismatch)
  - [Interpolation](#interpolation)
- [Feature Engineering](#feature-engineering)
- [Final Check](#final-check)
- [Export](#export)

## Imports and Loading

In [94]:
import pandas as pd
import re

In [95]:
df = pd.read_csv("data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 9999 rows, 9 columns


## Overview

In [96]:
display(df.head())

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


### Whitespace and Column Cleaning

In [97]:
# whitespace cleaning
cols = df.select_dtypes(include=["object", "str", "string"]).columns.tolist()

for col in cols:
    df[col] = df[col].str.replace(r"[\t\n]+", " ", regex=True).str.replace(r"\s{2,}", " ", regex=True).str.strip()

In [98]:
# column cleaning
print("Initial columns:", df.columns.tolist())
df.columns = df.columns.str.strip().str.lower().str.replace("-", "_")
print("Clean columns:", df.columns.tolist())

Initial columns: ['MOVIES', 'YEAR', 'GENRE', 'RATING', 'ONE-LINE', 'STARS', 'VOTES', 'RunTime', 'Gross']
Clean columns: ['movies', 'year', 'genre', 'rating', 'one_line', 'stars', 'votes', 'runtime', 'gross']


### Duplicate Handling

In [99]:
print("(1) Duplicates:", df.duplicated(subset=["movies", "one_line"]).sum())
display(df.loc[df.duplicated(subset=["movies", "one_line"], keep=False)].sort_values(by="movies").head(10))

(1) Duplicates: 857


,movies,year,genre,rating,one_line,stars,votes,runtime,gross
9992,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9989,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9990,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9443,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar | Stars: Aneurin Barna...,NaN,NaN,NaN
9991,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9982,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9980,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9981,1899,(2022– ),"Drama, History, Horror",NaN,Add a Plot,Director: Baran bo Odar,NaN,NaN,NaN
9177,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN
9178,800 metros,(2021– ),Documentary,NaN,Add a Plot,Director: León Siminiani,NaN,NaN,NaN


*Message for the client*

---

Hi,

I noticed the dataset doesn’t have a unique ID column. This makes it difficult to confidently identify duplicates.

In addition, there are no indications of what type the record is (Movie/Series). And if series, no season and episode number. This makes it even more difficult to say if two identical records are duplicates or different episodes of the same TV series. Hence, **I will proceed by assuming there are no duplicate rows and treating identical records distinctively**. Though, I’ll make sure everything is documented clearly.

Is this approach fine with you?

### Overall Insights

In [100]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movies    9999 non-null   str    
 1   year      9355 non-null   str    
 2   genre     9919 non-null   str    
 3   rating    8179 non-null   float64
 4   one_line  9999 non-null   str    
 5   stars     9999 non-null   str    
 6   votes     8179 non-null   str    
 7   runtime   7041 non-null   float64
 8   gross     460 non-null    str    
dtypes: float64(2), str(7)
memory usage: 703.2 KB


In [101]:
df.describe()

,rating,runtime
count,8179.000000,7041.000000
mean,6.921176,68.688539
std,1.220232,47.258056
min,1.100000,1.000000
25%,6.200000,36.000000
50%,7.100000,60.000000
75%,7.800000,95.000000
max,9.900000,853.000000


In [102]:
na_sum_before_formatting_and_conversion = df.isna().sum()

print(na_sum_before_formatting_and_conversion)

movies         0
year         644
genre         80
rating      1820
one_line       0
stars          0
votes       1820
runtime     2958
gross       9539
dtype: int64


## Data Formatting and Type Conversion

### Formatting Strings

In [103]:
str_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

print("String type columns:", str_cols)
display(df[str_cols].head(3))

String type columns: ['movies', 'year', 'genre', 'one_line', 'stars', 'votes', 'gross']


,movies,year,genre,one_line,stars,votes,gross
0,Blood Red Sky,(2021),"Action, Horror, Thriller",A woman with a mysterious illness is forced in...,Director: Peter Thorwarth | Stars: Peri Baumei...,"21,062",NaN
1,Masters of the Universe: Revelation,(2021– ),"Animation, Action, Adventure",The war for Eternia begins again in what may b...,"Stars: Chris Wood, Sarah Michelle Gellar, Lena...","17,870",NaN
2,The Walking Dead,(2010–2022),"Drama, Horror, Thriller",Sheriff Deputy Rick Grimes wakes up from a com...,"Stars: Andrew Lincoln, Norman Reedus, Melissa ...","885,805",NaN


The hyphens in entries of the year column are not the ordinary hyphens -, but instead an **En Dash** or **Em Dash** (UNICODE **\u2013** and **\u2014** respectively). The beneath cell replaces these unusual dashes with the common hyphen -.

In [104]:
df["year"] = df.year.str.replace("\u2013", "-").str.replace("\u2014", "-")

Now we proceed to correcting formats.

In [105]:
def format_masker(df: pd.DataFrame, col_format: dict, show=False):
    for col, format in col_format.items():
        mask = df[col].str.match(format)

        print(f"Non-matching values for column {col} and format {format}:", (~mask).sum())
        if show:
            display(df[col][(~mask)])
            print("-"*64)

col_format_pair = {
    "year": r"\(\d{4}\-?\s?(?:\d{4})?\)",
    }

format_masker(df, col_format_pair, show=True)

Non-matching values for column year and format \(\d{4}\-?\s?(?:\d{4})?\): 1692


36           (I) (2018- )
84          (II) (2007- )
125            (I) (2019)
155     (2021 TV Special)
161            (I) (2017)
              ...        
9909                  NaN
9910                  NaN
9911                  NaN
9912                  NaN
9913                  NaN
Name: year, Length: 1692, dtype: str

----------------------------------------------------------------


In [106]:
capture_pattern = r"(.*)(\(\d{4}\-?\s?(?:\d{4})?\))(.*)"

df["year"] = df.year \
    .str.replace(r"[a-zA-Z\s]+", "", regex=True) \
    .str.replace(capture_pattern, r"\2", regex=True)
    # remove letters and spaces anywhere in the text
    # replace the entire string with only the desired capture group


In [107]:
year_pattern = col_format_pair["year"]

year_mask = df.year.str.match(year_pattern, na=False) # na=False so existing nulls count as non-matching

df.loc[~year_mask, "year"] = pd.NA

year_matching_count = df[year_mask].year.shape[0]
year_non_matching_count = df[~year_mask].year.shape[0]
print("Matching values:", year_matching_count)
print("Non-matching values:", year_non_matching_count)
print("Total:", year_matching_count + year_non_matching_count)

Matching values: 9251
Non-matching values: 748
Total: 9999


The only column I checked the formatting is the year column. This is because the other columns are filled with a bunch of texts (no proper format or pattern / unstructured). I may check for extreme absurdity in each column but I prefer to assume there is no such a thing.

As a result, after formatting the year column, the number of NA values has increased from 644 to 748. Right now, it is guaranteed every valid data inside follows the format `\(\d{4}\-?\s?(?:\d{4})?\)`.

### Numerical Data Conversion

#### `gross` to Float64

In [108]:
valid_gross_values = df.gross.dropna().sort_values()

display(valid_gross_values)

1111     $0.00M
4637     $0.00M
4641     $0.00M
4648     $0.00M
4783     $0.00M
         ...   
793     $94.90M
383     $95.86M
262     $96.52M
582     $97.10M
421     $97.69M
Name: gross, Length: 460, dtype: str

Seems like every valid value follows the format `\$\d+\.\d{2}M`.

In [109]:
gross_mask = df.gross.str.fullmatch(r"\$\d+\.\d{2}M")

display(df.gross[gross_mask])

print(r"Every valid gross value follow the format \$\d+\.\d{2}M >>>", gross_mask.sum() == valid_gross_values.shape[0])

77       $75.47M
85      $402.45M
95       $89.22M
111     $315.54M
125      $57.01M
          ...   
5750      $0.09M
5770      $0.00M
5835      $0.01M
6056      $0.01M
6292      $0.14M
Name: gross, Length: 460, dtype: str

Every valid gross value follow the format \$\d+\.\d{2}M >>> True


In [110]:
def parse_gross(gross_str: str):
    if pd.isna(gross_str):
        return pd.NA
    
    s = gross_str.replace("$", "").replace("M", "")

    return float(s) * 1e6

df.gross = df.gross.apply(parse_gross).astype("Float64")

In [111]:
display(df.gross.dropna())

77       75470000.0
85      402450000.0
95       89220000.0
111     315540000.0
125      57010000.0
           ...     
5750        90000.0
5770            0.0
5835        10000.0
6056        10000.0
6292       140000.0
Name: gross, Length: 460, dtype: Float64

#### `votes` to Int64

In [112]:
df["votes"] = pd.to_numeric(df.votes.str.replace(",", "")).astype("Int64")

### Conclusion

In [113]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   movies    9999 non-null   str    
 1   year      9251 non-null   str    
 2   genre     9919 non-null   str    
 3   rating    8179 non-null   float64
 4   one_line  9999 non-null   str    
 5   stars     9999 non-null   str    
 6   votes     8179 non-null   Int64  
 7   runtime   7041 non-null   float64
 8   gross     460 non-null    Float64
dtypes: Float64(1), Int64(1), float64(2), str(5)
memory usage: 722.7 KB


In [114]:
na_sum_after_formatting_and_conversion = df.isna().sum()

na_comparison = pd.DataFrame({
    "before": na_sum_before_formatting_and_conversion,
    "after": na_sum_after_formatting_and_conversion,
    "change": na_sum_after_formatting_and_conversion - na_sum_before_formatting_and_conversion,
})

display(na_comparison)

,before,after,change
movies,0,0,0
year,644,748,104
genre,80,80,0
rating,1820,1820,0
one_line,0,0,0
stars,0,0,0
votes,1820,1820,0
runtime,2958,2958,0
gross,9539,9539,0


The NA count of year has increased by 104 values after data type conversion. This is due to the fact that there were valid but improper values before, such as (III), that got converted to NA.

## ROMI Approach

### Relation

No direct relation among columns.

### Outlier

In [115]:
df.describe()

,rating,votes,runtime,gross
count,8179.000000,8179.0,7041.000000,460.0
mean,6.921176,15124.062722,68.688539,43701869.565217
std,1.220232,70054.5777,47.258056,82423303.954897
min,1.100000,5.0,1.000000,0.0
25%,6.200000,166.0,36.000000,150000.0
50%,7.100000,789.0,60.000000,6145000.0
75%,7.800000,3772.0,95.000000,46947500.0
max,9.900000,1713028.0,853.000000,504010000.0


No numerical outlier.

### Mismatch

In [116]:
df.head(3)

,movies,year,genre,rating,one_line,stars,votes,runtime,gross
0,Blood Red Sky,(2021),"Action, Horror, Thriller",6.1,A woman with a mysterious illness is forced in...,Director: Peter Thorwarth | Stars: Peri Baumei...,21062,121.0,<NA>
1,Masters of the Universe: Revelation,(2021-),"Animation, Action, Adventure",5.0,The war for Eternia begins again in what may b...,"Stars: Chris Wood, Sarah Michelle Gellar, Lena...",17870,25.0,<NA>
2,The Walking Dead,(2010-2022),"Drama, Horror, Thriller",8.2,Sheriff Deputy Rick Grimes wakes up from a com...,"Stars: Andrew Lincoln, Norman Reedus, Melissa ...",885805,44.0,<NA>


In [117]:
year_counts = df.groupby("movies")["year"].nunique()

problem_movies = year_counts[year_counts > 1].index

if len(problem_movies) > 0:
    print("Mismatch found")
    print(problem_movies)
    display(df[df.movies.isin(problem_movies)].drop_duplicates(subset=["movies", "year"]).sort_values(by="movies"))
else:
    print("No mismatch found")

Mismatch found
Index(['Away', 'Bad Blood', 'Beauty and the Beast', 'Blood Brother',
       'Bodyguard', 'Boku dake ga inai machi', 'Fearless', 'Garbage',
       'Hagane no renkinjutsushi', 'Heartbreak High', 'Heist', 'Hit and Run',
       'Home', 'Jinn', 'Jonas', 'Kakegurui', 'Kingdom', 'Ludo',
       'Malibu Rescue', 'Mob Psycho 100', 'One of Us', 'Paranoid', 'Perdida',
       'Private Life', 'Revenge', 'Rosemary's Baby', 'Safe', 'Security',
       'Sexy Beasts', 'Snowpiercer', 'Tales of the City', 'The Oscars',
       'The Silence', 'The Stranger', 'The Whole Truth', 'Undercover',
       'Wanderlust'],
      dtype='str', name='movies')


,movies,year,genre,rating,one_line,stars,votes,runtime,gross
967,Away,(2020),"Drama, Romance, Sci-Fi",6.6,An American astronaut struggles with leaving h...,"Stars: Hilary Swank, Josh Charles, Vivian Wu, ...",22137,498.0,<NA>
2464,Away,(2016),"Crime, Drama",6.8,A story set in the north English seaside town ...,"Director: David Blair | Stars: Timothy Spall, ...",2079,105.0,<NA>
652,Bad Blood,(2017-),"Crime, Drama",7.5,A dramatization of the life and death of Montr...,"Stars: Kim Coates, Louis Ferreira, Sharon Tayl...",8238,47.0,<NA>
4246,Bad Blood,(2015),"Action, Drama, Thriller",4.7,"Lauren's life is on a positive trajectory, unt...","Director: Adam Silver | Stars: Taylor Cole, Bi...",402,87.0,<NA>
576,Beauty and the Beast,(1991),"Animation, Family, Fantasy",8.0,A prince cursed to spend his days as a hideous...,"Directors: Gary Trousdale, Kirk Wise | Stars: ...",425384,84.0,218970000.0
...,...,...,...,...,...,...,...,...,...
5454,The Whole Truth,(2009),"Action, Comedy",3.2,An acting coach makes it big ... Not in Hollyw...,Director: Colleen Patrick | Stars: Elisabeth R...,114,99.0,<NA>
4126,Undercover,(2021-),"Crime, Drama, Mystery",NaN,The story about a woman who joins an organized...,"Stars: Ahn Bo-Hyun, So-hee Han, Yull Jang, San...",<NA>,NaN,<NA>
611,Undercover,(2019-),"Crime, Drama, Thriller",7.8,"Inspired by real events, undercover agents inf...","Stars: Tom Waes, Frank Lammers, Manou Kersting...",12708,50.0,<NA>
5460,Wanderlust,(2006),"Documentary, History",6.6,A documentary on road movies and their effect ...,"Directors: Shari Springer Berman, Robert Pulci...",122,84.0,<NA>


The above cell aims to find a case where there exist different year records for the same movie, because I thought that would be a potential flaw. But when I checked it with the above cell, I found out such cases simply refer to different movies with the same title. So, no problem found here.

### Interpolation

No need for interpolation.

## Feature Engineering

**Feature Engineering** is the process of extracting new, meaningful information from existing columns.

For this dataset, I plan to:

- Extract features from `year`:
    - `type` — either "Movie" or "Series", based on how the year is formatted. `(2018)` with no hyphen indicates a movie; `(2018-)` with a trailing hyphen, or `(2018-2022)` with a full range, indicates a series that is still in production or has ended, respectively.
    - *[considered but omitted]* `progress` (Series only) — whether a series is still in production ("On") or has ended ("Ended"). Movies will have `NA` for this column.
    - `start_year` and `end_year` — the timeline of the title. Movies and series still in production have no `end_year` (`NA`).
- Split `stars` into `directors` and `stars`, keeping `stars` for cast names only.

In [118]:
# type
year_hyphen_mask = df.year.str.contains("-", na=False, regex=False)

df.loc[year_hyphen_mask, "type"] = "Series"
df.loc[(~year_hyphen_mask) & (df.year.notna()), "type"] = "Movie"

In [119]:
# start_year and end_year
year_parts = df["year"].str.extract(r"\((\d{4})-?\s?(\d{4})?\)")

df["start_year"] = pd.to_numeric(year_parts[0], errors="coerce").astype("Int64")
df["end_year"] = pd.to_numeric(year_parts[1], errors="coerce").astype("Int64")

In [120]:
# stars to directors and stars
df["directors"] = (
    df["stars"]
    .str.extract(r"Directors?:\s*([^|]+)", expand=False) # expand=False to make the return type Series and not DataFrame
    .str.strip()
)

df["stars"] = (
    df["stars"]
    .str.extract(r"Stars:\s*(.+)$", expand=False)
    .str.strip()
)

I skipped the idea of `progress` as its value can be deduced from `end_year` and `type`.

In [121]:
column_order = ["movies", "type", "start_year", "end_year", "genre", "rating", "one_line", "directors", "stars", "votes", "runtime", "gross"]

df = df[column_order]
df.head(3)

,movies,type,start_year,end_year,genre,rating,one_line,directors,stars,votes,runtime,gross
0,Blood Red Sky,Movie,2021,<NA>,"Action, Horror, Thriller",6.1,A woman with a mysterious illness is forced in...,Peter Thorwarth,"Peri Baumeister, Carl Anton Koch, Alexander Sc...",21062,121.0,<NA>
1,Masters of the Universe: Revelation,Series,2021,<NA>,"Animation, Action, Adventure",5.0,The war for Eternia begins again in what may b...,NaN,"Chris Wood, Sarah Michelle Gellar, Lena Headey...",17870,25.0,<NA>
2,The Walking Dead,Series,2010,2022,"Drama, Horror, Thriller",8.2,Sheriff Deputy Rick Grimes wakes up from a com...,NaN,"Andrew Lincoln, Norman Reedus, Melissa McBride...",885805,44.0,<NA>


## Final Check

In [122]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   movies      9999 non-null   str    
 1   type        9251 non-null   str    
 2   start_year  9251 non-null   Int64  
 3   end_year    1388 non-null   Int64  
 4   genre       9919 non-null   str    
 5   rating      8179 non-null   float64
 6   one_line    9999 non-null   str    
 7   directors   6353 non-null   str    
 8   stars       8615 non-null   str    
 9   votes       8179 non-null   Int64  
 10  runtime     7041 non-null   float64
 11  gross       460 non-null    Float64
dtypes: Float64(1), Int64(3), float64(2), str(6)
memory usage: 976.6 KB


In [123]:
df.describe()

,start_year,end_year,rating,votes,runtime,gross
count,9251.0,1388.0,8179.000000,8179.0,7041.000000,460.0
mean,2016.243974,2016.92219,6.921176,15124.062722,68.688539,43701869.565217
std,7.299851,5.239856,1.220232,70054.5777,47.258056,82423303.954897
min,1932.0,1969.0,1.100000,5.0,1.000000,0.0
25%,2015.0,2014.0,6.200000,166.0,36.000000,150000.0
50%,2018.0,2019.0,7.100000,789.0,60.000000,6145000.0
75%,2020.0,2020.0,7.800000,3772.0,95.000000,46947500.0
max,2023.0,2022.0,9.900000,1713028.0,853.000000,504010000.0


In [124]:
df.isna().sum()

movies           0
type           748
start_year     748
end_year      8611
genre           80
rating        1820
one_line         0
directors     3646
stars         1384
votes         1820
runtime       2958
gross         9539
dtype: int64

## Export

In [125]:
df.to_csv("data/clean.csv", index=False)
print("Saved successfully!")

Saved successfully!
